# 03 — Baselines: ItemKNN and logistic regression

Same date split, rows, scorer interface and evaluation as notebooks 01 / 02. **ItemKNN** (`implicit` cosine, item-item, K neighbours): history only; fitted on every train-window booking; cold rows (empty history) get the train-window first-booking popularity; ties broken by train popularity. **Logistic regression**: multinomial over the clusters on the one-hot of the same 14 query fields the query embedding reads (no history), fitted on every train-window booking, SAGA passes with the same early stopping as the neural models (`configs/train.yaml`). K and C are chosen on **autoregressive valid NDCG@10** with the one-SE rule by user (ties to the smaller K / smaller C); logistic regression runs for every seed. Logistic regression reads no history, so its autoregressive and teacher-forced ranks must be identical (asserted).

In [1]:
import json
import os
import sys
import time
from datetime import date
from pathlib import Path

import numpy as np
import torch  # torch before pandas-importing modules (Windows c10.dll)
import yaml

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]

from nbp.data.split import TRAIN
from nbp.eval import runner as R
from nbp.eval.report import metrics, row_set_masks, write_json
from nbp.paths import CONFIGS, EXPERIMENTS, INTERIM, PROCESSED

SPLIT = yaml.safe_load((CONFIGS / "split.yaml").read_text(encoding="utf-8"))
TRAIN_CFG = yaml.safe_load((CONFIGS / "train.yaml").read_text(encoding="utf-8"))
MODEL_CFG = yaml.safe_load((CONFIGS / "smlprec_expedia.yaml").read_text(encoding="utf-8"))
assert MODEL_CFG["MAX_ITEM_LIST_LENGTH"] == SPLIT["max_history"], "history length differs between configs"
SEEDS = list(SPLIT["seeds"])
REF_SEED = SEEDS[0]  # grids and sameDest weights are chosen on this seed
FAST = os.environ.get("NBP_FAST") == "1"  # smoke run: a user subsample, short training, outputs under *_fast
FAST_FRAC = float(os.environ.get("NBP_FAST_FRAC", "0.02"))
if FAST:
    TRAIN_CFG["max_epochs"] = int(os.environ.get("NBP_FAST_EPOCHS", "2"))
RES = ROOT / "results" / ("week6_retrain_fast" if FAST else "week6_retrain")
RANKS = INTERIM / ("week6_fast" if FAST else "week6")
EXP = EXPERIMENTS / "_fast" if FAST else EXPERIMENTS
TODAY = f"{date.today():%Y-%m-%d}"
K_MON = int(TRAIN_CFG["monitor"].split("@")[1])
BATCH = int(TRAIN_CFG["eval_batch"])
torch.set_num_threads(int(os.environ.get("NBP_THREADS", torch.get_num_threads())))

rows_full = R.load_rows(PROCESSED / "expedia_date_split" / "rows.npz")
codes = rows_full["codes"]  # (N, F) query codes, row = global event index
N_ITEMS = int(rows_full["y"].max()) + 1  # model ids 0 (padding) .. number of clusters
rows = R.subsample_users(rows_full, FAST_FRAC, 0) if FAST else rows_full
views = R.window_views(rows)
idx = R.window_index(rows)
Y = {w: rows["y"][i].astype(np.int64) for w, i in idx.items()}
L = {w: rows["L"][i] for w, i in idx.items()}
H = {w: rows["h"][i] for w, i in idx.items()}
USERS = {w: rows["user"][i] for w, i in idx.items()}
probe, y_probe = R.train_probe_view(rows, TRAIN_CFG["train_probe_rows"], TRAIN_CFG["train_probe_seed"])
print("FAST" if FAST else "FULL", {w: len(v) for w, v in views.items()}, "| items", N_ITEMS, "| threads", torch.get_num_threads())

FULL {'valid': 393399, 'test': 378788} | items 101 | threads 2


In [2]:
def summary_tables(ranks: dict) -> dict:
    """Metrics per window x protocol x row set (unrounded)."""
    out = {}
    for w in ("valid", "test"):
        masks = row_set_masks(L[w])
        for proto in ("ar", "tf"):
            out[f"{w}_{proto}"] = {rs: metrics(ranks[f"{w}_{proto}"], m) for rs, m in masks.items()}
    return out


def save_ranks(name: str, ranks: dict) -> Path:
    RANKS.mkdir(parents=True, exist_ok=True)
    path = RANKS / f"ranks_{name}.npz"
    np.savez_compressed(path, **{k: v for k, v in ranks.items()}, **{f"event_{w}": rows["event"][i] for w, i in idx.items()})
    return path

In [3]:
from nbp.baselines.itemknn import fit_itemknn, itemknn_scorer, user_item_matrix
from nbp.baselines.logreg import fit_encoder, fit_logreg, logreg_proba, logreg_scorer, train_log_loss
from nbp.eval.metrics import ndcg_at_k, target_rank
from nbp.hybrid.fusion import one_se_pick
from nbp.models.early_stop import mean_nll
from nbp.models.train import git_hash

if FAST and TRAIN_CFG["min_delta"] is None:
    TRAIN_CFG["min_delta"] = json.loads((RES / "min_delta_calibration.json").read_text(encoding="utf-8"))["min_delta"]
assert TRAIN_CFG["min_delta"] is not None, "run notebook 00 first (min_delta calibration)"


def per_row(r, k=K_MON):
    return (r <= k) / np.log2(r + 1)


tr = np.flatnonzero(rows["window"] == TRAIN)
y_tr = rows["y"][tr].astype(np.int64)
pop_share = np.bincount(y_tr, minlength=N_ITEMS) / len(y_tr)
first = tr[rows["L"][tr] == 0]
pop_first_share = np.bincount(rows["y"][first], minlength=N_ITEMS) / len(first)
print("train-window bookings", len(tr), "| first bookings", len(first))

train-window bookings 2180488 | first bookings 664727


## ItemKNN

In [4]:
ui = user_item_matrix(rows["user"][tr], y_tr, N_ITEMS)
knn_valid, sims = {}, {}
for k in TRAIN_CFG["itemknn_k_grid"]:
    sims[k] = fit_itemknn(ui, k)
    sc = R.score_view(views["valid"], itemknn_scorer(sims[k], pop_share, pop_first_share, TRAIN_CFG["itemknn_tie_eps"]), autoregressive=True, batch=BATCH)
    knn_valid[k] = target_rank(sc.scores, Y["valid"])
    print("K", k, "AR valid ndcg@10", round(per_row(knn_valid[k]).mean(), 4))
K_PICK, k_best, k_elig = one_se_pick({k: per_row(r) for k, r in knn_valid.items()}, USERS["valid"], complexity=lambda k: k)
print("chosen K", K_PICK, "| best", k_best, "| within one SE", k_elig)
knn_sf = itemknn_scorer(sims[K_PICK], pop_share, pop_first_share, TRAIN_CFG["itemknn_tie_eps"])
r_knn = R.evaluate(knn_sf, views, Y, BATCH)
save_ranks("itemknn", r_knn)
h0 = {w: H[w] == 0 for w in ("valid", "test")}
assert all(np.array_equal(r_knn[f"{w}_ar"][h0[w]], r_knn[f"{w}_tf"][h0[w]]) for w in h0), "ItemKNN: h = 0 rows must not depend on the protocol"
write_json(RES / "itemknn.json", {"model": "ItemKNN", "k_chosen": K_PICK, "k_best_on_valid": k_best, "k_within_one_se": k_elig,
           "valid_ar_ndcg@10_grid": {str(k): float(per_row(r).mean()) for k, r in knn_valid.items()},
           "fit_rows": "every train-window booking", "cold_rows": "train-window first-booking popularity",
           "git_commit": git_hash(ROOT), "split": SPLIT, "results": summary_tables(r_knn)})
t = summary_tables(r_knn)["test_ar"]
print(f"ItemKNN test AR all ndcg@10 {t['all']['ndcg@10']:.4f} warm {t['warm']['ndcg@10']:.4f} cold {t['cold']['ndcg@10']:.4f}")

C:\workspace\PROJECTS\next-best-product-recommendation\.venv\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.023930072784423828 seconds
  warnings.warn(


K 5 AR valid ndcg@10 0.1741


C:\workspace\PROJECTS\next-best-product-recommendation\.venv\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.025962114334106445 seconds
  warnings.warn(


K 10 AR valid ndcg@10 0.1756


C:\workspace\PROJECTS\next-best-product-recommendation\.venv\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.0229339599609375 seconds
  warnings.warn(


...

K 50 AR valid ndcg@10 0.1772


C:\workspace\PROJECTS\next-best-product-recommendation\.venv\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.03037095069885254 seconds
  warnings.warn(


K 100 AR valid ndcg@10 0.177


chosen K 50 | best 50 | within one SE [50]


ItemKNN test AR all ndcg@10 0.1825 warm 0.1990 cold 0.1118


## Logistic regression

In [5]:
enc = fit_encoder(codes[rows["event"][tr]])
X_tr = enc.transform(codes[rows["event"][tr]])
X_va = enc.transform(codes[rows["event"][idx["valid"]]])
probe_ev = probe.event
X_pr, y_pr = enc.transform(codes[probe_ev]), y_probe
print("train X", X_tr.shape)


def lr_eval(m):
    p = logreg_proba(m, X_va, N_ITEMS)
    r = target_rank(np.log(p + 1e-300), Y["valid"])
    return {"monitor": ndcg_at_k(r, K_MON), "valid_nll": mean_nll(np.log(p[:, 1:] + 1e-300), Y["valid"] - 1)}


def lr_fit(c, seed):
    return fit_logreg(X_tr, y_tr, c, seed, TRAIN_CFG["max_epochs"], TRAIN_CFG["patience"], TRAIN_CFG["min_delta"],
                      TRAIN_CFG["nll_guard_epochs"], lr_eval, lambda m: train_log_loss(m, X_pr, y_pr, N_ITEMS))


lr_grid = {}
for c in TRAIN_CFG["logreg_c_grid"]:
    m, prow, stopper, reason = lr_fit(c, REF_SEED)
    r = target_rank(np.log(logreg_proba(m, X_va, N_ITEMS) + 1e-300), Y["valid"])
    lr_grid[c] = {"model": m, "per_pass": prow, "best_pass": stopper.best_epoch, "stop_reason": reason, "valid_ar": r}
C_PICK, c_best, c_elig = one_se_pick({c: per_row(g["valid_ar"]) for c, g in lr_grid.items()}, USERS["valid"], complexity=lambda c: c)
print("chosen C", C_PICK, "| best", c_best, "| within one SE", c_elig)

train X (2180488, 33628)


C=0.1 seed=2022 pass 1: valid monitor 0.3669 nll 3.3170 | best 1, no gain 0/2


C=0.1 seed=2022 pass 2: valid monitor 0.3732 nll 3.2876 | best 2, no gain 0/2


C=0.1 seed=2022 pass 3: valid monitor 0.3748 nll 3.2804 | best 3, no gain 0/2


C=0.1 seed=2022 pass 4: valid monitor 0.3752 nll 3.2784 | best 3, no gain 1/2


...

C=10.0 seed=2022 pass 6: valid monitor 0.3885 nll 3.1677 | best 5, no gain 1/2


C=10.0 seed=2022 pass 7: valid monitor 0.3890 nll 3.1662 | best 7, no gain 0/2


C=10.0 seed=2022 pass 8: valid monitor 0.3892 nll 3.1655 | best 7, no gain 1/2


C=10.0 seed=2022 pass 9: valid monitor 0.3894 nll 3.1653 | best 7, no gain 2/2


chosen C 1.0 | best 1.0 | within one SE [1.0]


In [6]:
for seed in SEEDS:
    if seed == REF_SEED:
        g = lr_grid[C_PICK]
        m, prow, best_pass, reason = g["model"], g["per_pass"], g["best_pass"], g["stop_reason"]
    else:
        m, prow, stopper, reason = lr_fit(C_PICK, seed)
        best_pass = stopper.best_epoch
    r_lr = R.evaluate(logreg_scorer(m, enc, codes, N_ITEMS), views, Y, BATCH)
    for w in ("valid", "test"):
        assert np.array_equal(r_lr[f"{w}_ar"], r_lr[f"{w}_tf"]), "logistic regression must not depend on the protocol"
    save_ranks(f"logreg_s{seed}", r_lr)
    write_json(RES / f"logreg_s{seed}.json", {"model": "Logistic regression", "seed": seed, "c_chosen": C_PICK, "c_best_on_valid": c_best,
               "c_within_one_se": c_elig, "best_pass": best_pass, "stop_reason": reason, "per_pass": prow,
               "grid": {str(c): {"best_pass": g["best_pass"], "stop_reason": g["stop_reason"], "valid_ar_ndcg@10": float(per_row(g["valid_ar"]).mean())} for c, g in lr_grid.items()},
               "features": "one-hot of the 14 query fields (rows.npz codes), train-window vocabulary", "fit_rows": "every train-window booking",
               "git_commit": git_hash(ROOT), "split": SPLIT, "train_config": TRAIN_CFG, "results": summary_tables(r_lr)})
    t = summary_tables(r_lr)["test_ar"]
    print(f"seed {seed}: best pass {best_pass} ({reason}) | test AR all ndcg@10 {t['all']['ndcg@10']:.4f} warm {t['warm']['ndcg@10']:.4f} cold {t['cold']['ndcg@10']:.4f}")

seed 2022: best pass 8 (patience) | test AR all ndcg@10 0.3869 warm 0.3885 cold 0.3802


C=1.0 seed=2023 pass 1: valid monitor 0.3805 nll 3.2382 | best 1, no gain 0/2


C=1.0 seed=2023 pass 2: valid monitor 0.3901 nll 3.1727 | best 2, no gain 0/2


C=1.0 seed=2023 pass 3: valid monitor 0.3937 nll 3.1465 | best 3, no gain 0/2


C=1.0 seed=2023 pass 4: valid monitor 0.3955 nll 3.1322 | best 4, no gain 0/2


...

C=1.0 seed=2024 pass 7: valid monitor 0.3862 nll 3.1527 | best 6, no gain 1/2


C=1.0 seed=2024 pass 8: valid monitor 0.3866 nll 3.1500 | best 8, no gain 0/2


C=1.0 seed=2024 pass 9: valid monitor 0.3869 nll 3.1481 | best 8, no gain 1/2


C=1.0 seed=2024 pass 10: valid monitor 0.3871 nll 3.1467 | best 8, no gain 2/2


seed 2024: best pass 8 (patience) | test AR all ndcg@10 0.3811 warm 0.3823 cold 0.3759
